# Nelson Mandela Hybrid GraphRAG: Build

A real-world example of the hybrid graph pattern using Nelson Mandela data:

- **Unstructured**: Wikipedia biography → lexical-graph (entities, chunks, semantic embeddings)
- **Structured**: Events CSV + Organizations JSON → document-graph (typed nodes with dates, roles)

Both graphs share the same Neptune cluster and tenant, enabling cross-graph queries in Part 2.

## What This Demonstrates

Real knowledge graphs need both:
- **Unstructured text** (bios, articles, reports) → rich semantic context, entity discovery
- **Structured records** (databases, spreadsheets) → precise facts, dates, relationships

Separately, each is limited. Together, you can ask:
- "What key events shaped Mandela's life?" (semantic → structured correlation)
- "Show me the timeline of organizations Mandela was part of" (structured → traversal)
- "What does the bio say about the ANC?" (semantic search → entity linking)

**Prerequisites:**
- `pip install graphrag-toolkit-lexical-graph`
- `pip install graphrag-toolkit-document-graph[graphrag]`
- `pip install llama-index-core`
- Neptune cluster + OpenSearch Serverless (see 00-Setup)
- Sample data in `mandela_data/`


In [1]:
# Setup
import json, os
import pandas as pd
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory, VectorStoreFactory
from graphrag_toolkit.lexical_graph import LexicalGraphIndex
from graphrag_toolkit.document_graph.graph_build import node_to_cypher
from graphrag_toolkit.document_graph import Node

GRAPH_STORE = 'neptune-db://<your-neptune-cluster-endpoint>:8182'
VECTOR_STORE = '<your-opensearch-serverless-endpoint>'
TENANT = 'mandela'

graph_store = GraphStoreFactory.for_graph_store(GRAPH_STORE)
vector_store = VectorStoreFactory.for_vector_store(VECTOR_STORE)
gs = graph_store.__enter__()
print('Neptune + OpenSearch connected')


Neptune + OpenSearch connected


## Step 1: Build Lexical-Graph from Wikipedia

Index Mandela's biography into lexical-graph. This creates:
- **Entity nodes** (people, places, organizations) extracted by LLM
- **Chunk nodes** (text segments) with vector embeddings in OpenSearch
- **Relationships** between entities discovered from the text

After this step, you can ask semantic questions about Mandela's life.


In [2]:
from llama_index.core.schema import Document

# Static Mandela content
wiki_docs = [
    Document(
        text="""Nelson Rolihlahla Mandela was a South African anti-apartheid activist and politician who served as the first president of South Africa from 1994 to 1999. He was the country's first black head of state and the first elected in a fully representative democratic election. His government focused on dismantling the legacy of apartheid by fostering racial reconciliation. Mandela was born in Mvezo, Transkei, and studied law at the University of Fort Hare and the University of Witwatersrand. He joined the African National Congress (ANC) in 1943 and co-founded its Youth League. After the National Party came to power in 1948, he rose to prominence in the ANC's 1952 defiance campaign. He was arrested and imprisoned in 1962, and subsequently sentenced to life imprisonment for conspiring to overthrow the state. Mandela served 27 years in prison, split between Robben Island, Pollsmoor Prison, and Victor Verster Prison. An international campaign lobbied for his release, which was granted in 1990. He led negotiations with President F.W. de Klerk to abolish apartheid and establish multiracial elections in 1994, which he won.""",
        metadata={'source': 'wikipedia', 'title': 'Nelson Mandela'}
    ),
    Document(
        text="""Apartheid was a system of institutionalised racial segregation that existed in South Africa and South West Africa from 1948 to the early 1990s. Under apartheid, the rights of the majority non-white inhabitants were curtailed and white supremacy and Afrikaner minority rule were maintained. The National Party implemented apartheid as state policy after winning the 1948 general election. This racial segregation was enforced through legislation. People were classified into racial groups and separated by laws that banned social contact, mixed marriages, and established separate public facilities. The system was dismantled through negotiations between 1990 and 1993 and elections in 1994.""",
        metadata={'source': 'wikipedia', 'title': 'Apartheid'}
    ),
    Document(
        text="""The African National Congress (ANC) is the Republic of South Africa's governing political party. It has been the ruling party of post-apartheid South Africa since the election of Nelson Mandela in 1994. Founded in 1912 as the South African Native National Congress, its primary mission was to bring all Africans together as one people to defend their rights and freedoms. The ANC was banned in 1960 after the Sharpeville massacre and operated underground and in exile for 30 years until it was unbanned in 1990. Key figures include Nelson Mandela, Oliver Tambo, Walter Sisulu, and Albert Luthuli.""",
        metadata={'source': 'wikipedia', 'title': 'African National Congress'}
    ),
]

print(f'Created {len(wiki_docs)} documents')
for doc in wiki_docs:
    title = doc.metadata['title']
    print(f'  {title}: {len(doc.text)} chars')


Created 3 documents
  Nelson Mandela: 1125 chars
  Apartheid: 690 chars
  African National Congress: 596 chars


In [3]:
# Index into lexical-graph
os.environ['BATCH_WRITES_ENABLED'] = 'false'

print(f'Indexing {len(wiki_docs)} documents into lexical-graph...')
graph_index = LexicalGraphIndex(graph_store, vector_store)
graph_index.extract_and_build(wiki_docs, show_progress=True)
print('Lexical-graph build complete')


Indexing 3 documents into lexical-graph...


Extracting propositions [nodes: 1, num_workers: 4]: 100%|██████████| 1/1 [00:04<00:00,  4.48s/it]
Extracting propositions [nodes: 2, num_workers: 4]: 100%|██████████| 2/2 [00:06<00:00,  3.12s/it]
Extracting topics [nodes: 2, num_workers: 4]: 100%|██████████| 2/2 [00:15<00:00,  7.66s/it]
Building graph [batch_writes_enabled: False, batch_write_size: 25]: 100%|██████████| 46/46 [00:15<00:00,  2.90it/s]]
Building vector index [batch_writes_enabled: False, batch_write_size: 25]: 100%|██████████| 46/46 [00:01<00:00, 31.81it/s]
Building graph [batch_writes_enabled: False, batch_write_size: 25]: 100%|██████████| 124/124 [00:31<00:00,  3.92it/s]
Building vector index [batch_writes_enabled: False, batch_write_size: 25]: 100%|██████████| 124/124 [00:07<00:00, 16.51it/s]


Lexical-graph build complete


## Step 2: Build Document-Graph from Structured Data

Load structured records (events with dates, organizations with roles) as typed nodes.
These provide precise, queryable facts that complement the semantic layer:
- `Event` nodes: date, location, description, significance
- `Organization` nodes: name, role, founding date, members

Both use the same tenant so they coexist in one Neptune cluster.


In [4]:
def flatten_props(props):
    flat = {}
    for k, v in props.items():
        if isinstance(v, list):
            flat[k] = ', '.join(str(x) for x in v)
        elif isinstance(v, dict):
            flat[k] = json.dumps(v)
        elif v is None:
            continue
        else:
            flat[k] = v
    return flat

# Load Mandela structured data
events = pd.read_csv('mandela_data/mandela_events.csv')
orgs = json.load(open('mandela_data/research_papers.json'))
if isinstance(orgs, dict):
    key = list(orgs.keys())[0]
    orgs = orgs[key]

print(f'Events: {len(events)} rows')
print(f'Organizations/Papers: {len(orgs)} records')

# Write events to Neptune
for _, row in events.iterrows():
    record = row.to_dict()
    rid = str(record.get('id', record.get('event_id', hash(str(record)))))
    node = Node(id=rid, labels=['Event'], properties=flatten_props(record))
    cypher, params = node_to_cypher(node, tenant_id=TENANT)
    gs.execute_query(cypher, params)

# Write orgs/papers to Neptune
for record in orgs:
    rid = str(record.get('id', record.get('org_id', hash(str(record)))))
    node = Node(id=rid, labels=['Organization'], properties=flatten_props(record))
    cypher, params = node_to_cypher(node, tenant_id=TENANT)
    gs.execute_query(cypher, params)

print(f'\nWrote {len(events)} Event + {len(orgs)} Organization nodes to Neptune')


Events: 9 rows
Organizations/Papers: 5 records

Wrote 9 Event + 5 Organization nodes to Neptune


## Step 3: Verify

Confirm both graphs were built successfully in the same Neptune cluster.
Both lexical-graph entities and document-graph typed nodes should be present.


In [5]:
# Verify mandela tenant nodes
print("Mandela tenant nodes:")
for ntype in ["Event", "Organization"]:
    label = f"__{ntype}__{TENANT}__"
    result = gs.execute_query(f"MATCH (n:`{label}`) RETURN count(n) as cnt")
    cnt = result[0]["cnt"] if result else 0
    print(f"  {ntype}: {cnt}")

print("Build complete. Ready for 08b (Query).")


Mandela tenant nodes:
  Event: 9
  Organization: 5
Build complete. Ready for 08b (Query).
